## Code is shared by yunzhe. Here I set subpriority=0 for comparison with that from network flow method. --05-13-2026

In [1]:
import numpy as np
from functools import lru_cache
from astropy.table import Table, vstack
from scipy.interpolate import interp1d
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix
from scipy.spatial import KDTree
import sys
from utils import get_fiberpos
from parameters import (
    COLLISION_SEPARATION_ARCSEC,
    COLLISION_SEPARATION_DEG,
    )

In [2]:
@lru_cache(maxsize = None)
def load_platescale(telescope):
    #infile = f'spec/{telescope}_platescale.txt'
    infile = dir_root + "platescale.txt"
    columns = [
        ("radius", "f8"),
        ("theta", "f8"),
        ("radial_platescale", "f8"),
        ("az_platescale", "f8"),
        ("arclength", "f8"),
    ]
    try:
        _platescale = np.loadtxt(infile, usecols = [0, 1, 6, 7, 8], dtype = columns)
    except (IndexError, ValueError):
        _platescale = np.loadtxt(infile, usecols = [0, 1, 6, 7, 7], dtype = columns)
        #rzs = Table.read(f'spec/{telescope}_rzsn.txt', format = "ascii")
        rzs = Table.read(dir_root + '/rzsn.txt', format = "ascii")
        arclength = interp1d(rzs["R"], rzs["S"], kind = "quadratic")
        _platescale["arclength"] = arclength(_platescale["radius"])

    return _platescale

def get_radius_mm(theta, telescope):
    platescale = load_platescale(telescope)
    fn = interp1d(platescale['theta'], platescale['radius'], kind = 'quadratic', 
                  bounds_error = False, fill_value = 'extrapolate')
    radius = fn(theta)
    if(np.isscalar(theta)):
        return float(radius)
    else:
        return radius
    
def get_radius_deg(x, y, telescope):
    if not np.isscalar(x):
        x = np.asarray(x)
    if not np.isscalar(y):
        y = np.asarray(y)
    radius = np.sqrt(x **2 + y **2)
    platescale = load_platescale(telescope)
    fn = interp1d(platescale['radius'], platescale['theta'], kind = 'quadratic', 
                  bounds_error = False, fill_value = 'extrapolate')
    degree = fn(radius).astype(float)
    return degree

def patrol_radius_deg(x, y, telescope, patrol_radius_mm = 6.0):
    ps = load_platescale(telescope)
    R = np.hypot(x, y)
    fn_rad = interp1d(ps['radius'], ps['radial_platescale'], kind = 'quadratic', fill_value = 'extrapolate')
    radial_ps = fn_rad(R)
    patrol_deg = (radial_ps / 3600.0) * patrol_radius_mm
    return patrol_deg

def xy2radec(telra, teldec, x, y, telescope):
    r_rad = np.radians(get_radius_deg(x, y, telescope))
    q_rad = np.arctan2(y, x)

    x1, y1 = np.cos(r_rad), -np.sin(r_rad)
    v2 = np.stack([x1, y1 * np.cos(q_rad), -y1 * np.sin(q_rad)])

    rarotate = np.zeros(shape = (3, 3))
    telra_rad = np.radians(telra)
    rarotate[0] = [np.cos(telra_rad), -np.sin(telra_rad), 0]
    rarotate[1] = [np.sin(telra_rad), np.cos(telra_rad), 0]
    rarotate[2] = [0, 0, 1]

    decrotate = np.zeros(shape = (3, 3))
    teldec_rad = np.radians(teldec)
    decrotate[0] = [np.cos(teldec_rad), 0, -np.sin(teldec_rad)]
    decrotate[1] = [0, 1, 0]
    decrotate[2] = [np.sin(teldec_rad), 0, np.cos(teldec_rad)]

    x3, y3, z3 = rarotate.dot(decrotate.dot(v2))
    ra_deg = np.degrees(np.arctan2(y3, x3)) % 360
    dec_deg = np.degrees((np.pi / 2) - np.arccos(z3))
    return np.vstack((ra_deg, dec_deg)).T

def radec2xy(telra, teldec, ra, dec, telescope):
    inc = 90 - dec
    x0 = np.sin(np.radians(inc)) * np.cos(np.radians(ra))
    y0 = np.sin(np.radians(inc)) * np.sin(np.radians(ra))
    z0 = np.cos(np.radians(inc))
    coord = [x0, y0, z0]

    rarotate = np.zeros(shape = (3, 3))
    telra_rad = np.radians(telra)
    rarotate[0] = [np.cos(telra_rad), np.sin(telra_rad), 0]
    rarotate[1] = [-np.sin(telra_rad), np.cos(telra_rad), 0]
    rarotate[2] = [0, 0, 1]

    decrotate = np.zeros(shape = (3, 3))
    teldec_rad = np.radians(teldec)
    decrotate[0] = [np.cos(teldec_rad), 0, np.sin(teldec_rad)]
    decrotate[1] = [0, 1, 0]
    decrotate[2] = [-np.sin(teldec_rad), 0, np.cos(teldec_rad)]

    coord1 = np.matmul(rarotate, coord)
    coord2 = np.matmul(decrotate, coord1)
    x, y, z = coord2[0], coord2[1], coord2[2]

    newtelra, newteldec = 0, 0
    ra_rad = np.arctan2(y, x)
    dec_rad = (np.pi / 2) - np.arccos(z / np.sqrt((x **2) + (y **2) + (z **2)))
    radius_deg = np.degrees(2 * np.arcsin(np.sqrt((np.sin((dec_rad - newteldec) / 2) **2) + ((np.cos(newteldec)) * np.cos(dec_rad) * (np.sin((ra_rad - newtelra) / 2) **2)))))

    q_rad = np.arctan2(z, -y)
    radius_mm = get_radius_mm(radius_deg, telescope)
    x_focalplane = radius_mm * np.cos(q_rad)
    y_focalplane = radius_mm * np.sin(q_rad)
    return np.vstack((x_focalplane, y_focalplane)).T

def _radec_unit(coord):
    coord = np.atleast_2d(coord)
    ra = np.radians(coord[:, 0])
    dec = np.radians(coord[:, 1])
    return np.column_stack([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])

def _chord_radius(theta_deg):
    return 2.0 * np.sin(0.5 * np.radians(theta_deg))

def _get_tile_cands(coord_fiber, coord_tile, coord_target, r_patrol, telescope = 'just'):
    coord_tile = np.atleast_2d(coord_tile)
    num_fiber = coord_fiber.shape[0]

    r_edge = np.nanmax(np.hypot(coord_fiber[:, 0], coord_fiber[:, 1])) + r_patrol
    r_sky = float(get_radius_deg(r_edge, 0.0, telescope))

    tree_sky = KDTree(_radec_unit(coord_target))
    xyz_tile = _radec_unit(coord_tile)

    cand_lists, tile_of_fiber = [], []
    tile_near, tile_xy, tile_tree, tile_ind = [], [], [], []

    for i, tile in enumerate(coord_tile):
        near = np.array(tree_sky.query_ball_point(xyz_tile[i], r = _chord_radius(r_sky)), dtype = int)
        tile_near.append(near)

        if len(near) == 0:
            xy = np.empty((0, 2))
            tree = None
            ind = {}
            cands_local = [[] for _ in range(num_fiber)]
        else:
            xy = radec2xy(tile[0], tile[1], coord_target[near, 0], coord_target[near, 1], telescope)
            tree = KDTree(xy)
            ind = {int(t): j for j, t in enumerate(near)}
            cands_local = tree.query_ball_point(coord_fiber, r = r_patrol)

        tile_xy.append(xy)
        tile_tree.append(tree)
        tile_ind.append(ind)

        for cands in cands_local:
            if len(cands) == 0:
                cand_lists.append([])
            else:
                cand_lists.append(near[np.array(cands, dtype = int)].tolist())
            tile_of_fiber.append(i)

    return cand_lists, np.array(tile_of_fiber, dtype = int), tile_near, tile_xy, tile_tree, tile_ind

def ass_greedy(coord_fiber, coord_tile, coord_target, priorities, subpriorities, r_patrol = 6.0, r_exclude = 2.0):
    coord_tile = np.atleast_2d(coord_tile)
    num_targets = coord_target.shape[0]

    ass_mask = np.zeros(num_targets, dtype = bool)
    pat_mask = np.zeros(num_targets, dtype = bool)
    noo_tile = [set() for _ in range(len(coord_tile))]

    cand_lists, tile_of_fiber, tile_near, tile_xy, tile_tree, tile_ind = _get_tile_cands(coord_fiber, coord_tile, coord_target, r_patrol)
    fiber_order = np.argsort([-len(cands) for cands in cand_lists])

    for f in fiber_order:
        cands = cand_lists[f]
        if not cands:
            continue
        
        i = tile_of_fiber[f]
        pat_mask[cands] = True
        avail = [t for t in cands if (not ass_mask[t]) and (t not in noo_tile[i])]
        if not avail:
            continue

        pr = priorities[avail]
        spr = subpriorities[avail]
        idx_best = avail[np.lexsort((spr, pr))[-1]]
        ass_mask[idx_best] = True

        xy_best = tile_xy[i][tile_ind[i][idx_best]]
        ii = tile_tree[i].query_ball_point(xy_best, r = r_exclude)
        noo_tile[i].update(tile_near[i][np.array(ii, dtype = int)].tolist())

    return ass_mask, pat_mask

def ass_milp(coord_fiber, coord_tile, coord_target, priorities, subpriorities, r_patrol = 6.0, r_exclude = 2.0):
    coord_tile = np.atleast_2d(coord_tile)
    num_targets = coord_target.shape[0]

    ass_mask = np.zeros(num_targets, dtype = bool)
    pat_mask = np.zeros(num_targets, dtype = bool)

    cand_lists, tile_of_fiber, tile_near, tile_xy, tile_tree, tile_ind = _get_tile_cands(coord_fiber, coord_tile, coord_target, r_patrol)

    edge_f, edge_t = [], []
    tile_target_edges = [{} for _ in range(len(coord_tile))]

    for f, cands in enumerate(cand_lists):
        if not cands:
            continue

        i = tile_of_fiber[f]
        pat_mask[cands] = True

        for t in cands:
            e = len(edge_t)
            edge_f.append(f)
            edge_t.append(t)
            tile_target_edges[i].setdefault(t, []).append(e)

    edge_f = np.array(edge_f, dtype = int)
    edge_t = np.array(edge_t, dtype = int)
    nedge = len(edge_t)

    if nedge == 0:
        return ass_mask, pat_mask

    rows, cols, data, lb, ub = [], [], [], [], []
    row = 0

    fiber_edges = {}
    for e, f in enumerate(edge_f):
        fiber_edges.setdefault(f, []).append(e)

    for f, ii in fiber_edges.items():
        rows.extend([row] * len(ii)); cols.extend(ii); data.extend([1.0] * len(ii))
        lb.append(0.0); ub.append(1.0); row += 1

    target_edges = {}
    for e, t in enumerate(edge_t):
        target_edges.setdefault(t, []).append(e)

    for t, ii in target_edges.items():
        rows.extend([row] * len(ii)); cols.extend(ii); data.extend([1.0] * len(ii))
        lb.append(0.0); ub.append(1.0); row += 1

    for i in range(len(coord_tile)):
        t_arr = np.array(list(tile_target_edges[i].keys()), dtype = int)
        if len(t_arr) < 2:
            continue

        j_arr = np.array([tile_ind[i][t] for t in t_arr], dtype = int)
        tree_local = KDTree(tile_xy[i][j_arr])

        for i1, i2 in tree_local.query_pairs(r = r_exclude):
            t1, t2 = t_arr[i1], t_arr[i2]
            ii = tile_target_edges[i][t1] + tile_target_edges[i][t2]
            rows.extend([row] * len(ii)); cols.extend(ii); data.extend([1.0] * len(ii))
            lb.append(0.0); ub.append(1.0); row += 1

    A = coo_matrix((data, (rows, cols)), shape = (row, nedge)).tocsr()
    pr = priorities[edge_t].astype(float)
    spr = subpriorities[edge_t].astype(float)
    # pr = (pr - np.nanmin(pr)) / (np.nanmax(pr) - np.nanmin(pr) + 1e-30)
    # spr = (spr - np.nanmin(spr)) / (np.nanmax(spr) - np.nanmin(spr) + 1e-30)
    #res = milp(c = -(1.0 + 1.e-6 * pr + 1.e-9 * spr), integrality = np.ones(nedge), bounds = Bounds(0, 1), constraints = LinearConstraint(A, np.array(lb), np.array(ub)))
    res = milp(c = -(10.0+10*pr+spr), integrality = np.ones(nedge), bounds = Bounds(0, 1), constraints = LinearConstraint(A, np.array(lb), np.array(ub)))
    if not res.success:
        raise RuntimeError(res.message)

    ass_mask[edge_t[res.x > 0.5]] = True
    return ass_mask, pat_mask

In [3]:
# Get fiber positions and set random seed
fiberpos_xy = get_fiberpos()
N_fibers = fiberpos_xy.shape[0]
print(f"Number of fibers: {N_fibers}")

rmagcut = 20.5
ifile = dir_root + "lightcone_0_0_0_rmagcut20.5_20ra25_1dec5.fits"
gal_cat = Table.read(ifile)

ifile = dir_root + "tiles_1passe_20ra25_1dec5.fits"
tiles = Table.read(ifile)

rand_seed = 23


Number of fibers: 2184


In [4]:
gal_cat

TARGETID,RA,DEC,PRIORITY,SUBPRIORITY
int64,float64,float64,float32,float64
1710,24.92839291640655,3.0034938549464716,1.0,0.6939330806573643
3050,21.07504376835192,4.161448823209443,1.0,0.6414582208782306
3119,22.416492045715987,2.8095594024448824,1.0,0.12864422431763622
3194,24.30791333302096,3.7892279069605275,1.0,0.11370805013132934
3284,22.53705711495348,2.898830612269199,1.0,0.6533455213345873
3298,23.366943935780345,4.285265393659301,1.0,0.8534571059651607
3604,24.40196885685071,4.227160437593372,1.0,0.2017791344404063
3678,21.78931250516452,3.50653503141687,1.0,0.218018637085736
3705,21.80215522519243,1.330821224815287,1.0,0.7165846359235828


In [5]:
# Build inputs for ass_milp (same convention as ass_greedy: fiber xy in mm, tile/target RA,DEC in deg)
coord_target = np.column_stack([np.asarray(gal_cat["RA"]), np.asarray(gal_cat["DEC"])])
priorities = np.asarray(gal_cat["PRIORITY"], dtype=float)
subpriorities = np.asarray(gal_cat["SUBPRIORITY"], dtype=float)
##subpriorities = np.zeros(len(gal_cat), dtype=float)

In [6]:
ass_mask_list = []
for i in [2,6]:
    coord_tile = np.array([[float(tiles["RA"][i]), float(tiles["DEC"][i])]])
    
    ass_mask, pat_mask = ass_milp(
        fiberpos_xy,
        coord_tile,
        coord_target,
        priorities,
        subpriorities,
        r_patrol=6.0,
        r_exclude=2.0,
    )
    print(
        f"assigned={ass_mask.sum()}, "
        f"within patrol reach={pat_mask.sum()}"
    )
    ass_mask_list.append(ass_mask)

assigned=1645, within patrol reach=3171
assigned=1791, within patrol reach=4162


In [7]:
# Extra global validation using healpy:
# ensure any two assigned targets are separated by >= COLLISION_SEPARATION_DEG.
import healpy as hp
from collections import defaultdict
def count_close_pairs_healpy(assigned_target_ids, gal_cat, separation_deg, nside=2048):
    """Return violating pairs (tid_i, tid_j, sep_deg) with sep < separation_deg."""
    if len(assigned_target_ids) < 2:
        return []

    ids = np.asarray(assigned_target_ids)
    ra = np.array([gal_cat["RA"][int(tid)] for tid in ids], dtype=np.float64)
    dec = np.array([gal_cat["DEC"][int(tid)] for tid in ids], dtype=np.float64)

    theta = np.radians(90.0 - dec)
    phi = np.radians(ra)
    pix = hp.ang2pix(nside, theta, phi)
    vecs = np.vstack(hp.ang2vec(theta, phi))

    pix_to_idx = defaultdict(list)
    for i, p in enumerate(pix):
        pix_to_idx[int(p)].append(i)

    cos_th = np.cos(np.radians(separation_deg))
    viol = []

    for i in range(len(ids)):
        v = vecs[i]
        cand_pix = hp.query_disc(nside, v, np.radians(separation_deg), inclusive=True)

        for cp in cand_pix:
            for j in pix_to_idx.get(int(cp), []):
                if j <= i:
                    continue
                # Great-circle separation via unit-vector dot product.
                dot = float(np.dot(v, vecs[j]))
                if dot > cos_th:
                    sep = np.degrees(np.arccos(np.clip(dot, -1.0, 1.0)))
                    if sep < separation_deg:
                        viol.append((int(ids[i]), int(ids[j]), float(sep)))

    return viol

In [8]:
for i, ass_mask in enumerate(ass_mask_list):
    gal_cat_assigned = gal_cat[ass_mask]
    healpy_viol = count_close_pairs_healpy(
    np.arange(len(gal_cat_assigned)),
    gal_cat_assigned,
    COLLISION_SEPARATION_DEG,
)
    print(
        f"Healpy global separation check: {len(healpy_viol)} violating pair(s) "
        f"with sep < {COLLISION_SEPARATION_DEG:.6f} deg"
    )
    if healpy_viol:
        print("  First 10 violating pairs (target_i, target_j, sep_deg):")
        for row in healpy_viol[:10]:
            print("   ", row)

    ofile = f"./milp_fba_tile{i}.fits"
    gal_cat_assigned.write(ofile, overwrite=True)

Healpy global separation check: 2 violating pair(s) with sep < 0.004340 deg
  First 10 violating pairs (target_i, target_j, sep_deg):
    (149, 1370, 0.004325262627305392)
    (473, 1467, 0.004302363806117207)
Healpy global separation check: 0 violating pair(s) with sep < 0.004340 deg


In [9]:
gal_milp_ass = vstack((Table.read("./milp_fba_tile0.fits"), Table.read("./milp_fba_tile1.fits")))
gal_milp_ass

TARGETID,RA,DEC,PRIORITY,SUBPRIORITY
int64,float64,float64,float32,float64
11042,23.672614972258927,3.8057942834231584,1.0,0.8513410130861883
26893,23.637348282373168,3.2080680810450795,1.0,0.5181753241019371
46135,23.467765306710746,3.788289190162148,1.0,0.829194124866474
48375,23.438687883783054,3.2625200911124668,1.0,0.9518777921177327
27379682,23.51686194722845,3.5206286936859437,1.0,0.5082719492569345
27381665,23.099911161380156,4.064178517959291,1.0,0.751464145369962
27382152,22.766379993443802,3.6159634479788703,1.0,0.5922898393350682
27383325,23.704392052104556,3.8035352187176197,1.0,0.6941270255450048
27383326,23.45429025589117,3.5592332488865543,1.0,0.6731780733313983


In [10]:
gal_flow_ass = Table.read("./nwf_fba.fits")
gal_flow_ass

TARGETID,RA,DEC,PRIORITY,SUBPRIORITY
int64,float64,float64,float32,float64
3119,22.416492045715987,2.8095594024448824,1.0,0.12864422431763622
3284,22.53705711495348,2.898830612269199,1.0,0.6533455213345873
26893,23.637348282373168,3.2080680810450795,1.0,0.5181753241019371
48375,23.438687883783054,3.2625200911124668,1.0,0.9518777921177327
54808,23.312989786026726,2.7163206498045533,1.0,0.4508711364919724
27376338,23.115484589397596,2.5348468711801635,1.0,0.179587818857323
27379682,23.51686194722845,3.5206286936859437,1.0,0.5082719492569345
27381149,23.326935126775332,2.5167227138810118,1.0,0.4868641761375153
27381665,23.099911161380156,4.064178517959291,1.0,0.751464145369962


In [11]:
np.sum(gal_flow_ass["PRIORITY"]+gal_flow_ass["SUBPRIORITY"]) - np.sum(gal_milp_ass["PRIORITY"] + gal_milp_ass["SUBPRIORITY"])

np.float64(-400.98330270430415)